<a href="https://colab.research.google.com/github/yuvika2411/MoodCanvas_AI/blob/main/MoodCanvas_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MoodCanvas AI
**Emotion-Aware Motivational Poster Generation**

**Run:** Upload this file to Google Colab → choose **Runtime > Change runtime type > T4 GPU** → run each cell in order.

**Idea:** Type how you feel. A pretrained transformer detects emotion. A pretrained diffusion model creates matching scenery. Python adds a motivational sentence to make a poster.


## 1. Install libraries
Transformers reads the mood. Diffusers generates an image. Pillow adds readable words.

In [ ]:
%pip -q install transformers==4.51.3 diffusers==0.33.1 accelerate==1.6.0 Pillow==11.2.1 scikit-learn==1.6.1
print('Project libraries installed successfully.')

## 2. Load the two pretrained models
**DistilRoBERTa:** a smaller transformer already trained to recognize seven emotions.
**SD-Turbo:** a diffusion model that generates a picture from a description.

The emotion model's authors trained it on English text from six datasets, using 2,811 examples per emotion and an 80/20 split. That is the authors' training, not training performed in this project. Source: https://huggingface.co/j-hartmann/emotion-english-distilroberta-base


In [ ]:
import torch, time, json
from transformers import pipeline
from diffusers import AutoPipelineForText2Image
from PIL import Image, ImageDraw, ImageFont, ImageOps
from IPython.display import display
from google.colab import files

assert torch.cuda.is_available(), 'Choose Runtime > Change runtime type > T4 GPU, then rerun.'
emotion_model = pipeline('text-classification',
    model='j-hartmann/emotion-english-distilroberta-base', device=-1)
image_model = AutoPipelineForText2Image.from_pretrained(
    'stabilityai/sd-turbo', torch_dtype=torch.float16, variant='fp16').to('cuda')
print('Both models are ready.')


## 2A. Basic emotion-model fine-tuning
We continue training the **last transformer layer and classification head**; earlier layers stay frozen. This is partial fine-tuning, not training from scratch. SD-Turbo is not fine-tuned.

- Training data: **28 manually written sentences**, four per emotion.
- Evaluation: the separate **14 sentences** in Section 5, never used for weight updates.
- Settings: **3 epochs**, batch size **4**, AdamW learning rate **2e-5**, maximum **128 tokens**.
- Training runs on CPU to avoid sharing GPU memory with the image model. It may take a few minutes.
- These tiny examples demonstrate training mechanics, not proven improvement. Their independence from the original model's pretraining data cannot be guaranteed.

Run all cells from the top. This cell starts from the original checkpoint on each run. It prints before/after evaluation later, epoch losses, and evidence that weights changed. No new training results are claimed until you execute it.


In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, set_seed
from torch.utils.data import DataLoader

set_seed(42)
checkpoint = 'j-hartmann/emotion-english-distilroberta-base'
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint).to('cpu')
# Keep a baseline for the same held-out test in Section 5.
baseline_emotion_model = emotion_model

training_examples = {'sadness': ['I feel down because my plans did not work out.', 'I miss my family and feel unhappy.', 'Losing my pet has left me feeling miserable.', 'I feel disappointed and want to cry.'], 'joy': ['I am thrilled that I got selected for the internship.', 'Spending time with my family fills me with happiness.', 'I feel proud and excited after finishing my project.', 'Today has been a wonderful and joyful day.'], 'anger': ['I am furious about being treated unfairly.', 'Their rude behavior makes me extremely angry.', 'I feel irritated because nobody listened to me.', 'I am frustrated that my hard work was ignored.'], 'fear': ['I am worried and frightened about the interview.', 'Walking alone in that dark street scares me.', 'I feel anxious about failing my examination.', 'The sudden threat made me afraid.'], 'surprise': ['I was amazed when my friends arrived unexpectedly.', 'What a surprise to see you here!', 'I cannot believe this unexpected announcement.', 'I am astonished that we won the competition.'], 'disgust': ['The smell from the overflowing bin is revolting.', 'That moldy bread makes me feel disgusted.', 'I find the dirty bathroom repulsive.', 'Seeing insects in my meal makes me sick.'], 'neutral': ['The class is scheduled for Monday morning.', 'There are four chairs in this room.', 'I uploaded the document to the shared folder.', 'The bus route has twelve stops.']}

train_texts, train_labels = [], []
# Use the checkpoint's label IDs instead of assuming a class order.
label_to_id = {name.lower(): int(idx) for idx, name in model.config.id2label.items()}
for label, sentences in training_examples.items():
    for sentence in sentences:
        train_texts.append(sentence)
        train_labels.append(label_to_id[label])
encoded = tokenizer(train_texts, padding=True, truncation=True,
                    max_length=128, return_tensors='pt')
train_data = [{**{key: value[i] for key, value in encoded.items()},
               'labels': torch.tensor(train_labels[i], dtype=torch.long)}
              for i in range(len(train_texts))]
loader = DataLoader(train_data, batch_size=4, shuffle=True)

# Freeze early layers. Update the last encoder block and classifier only.
model.requires_grad_(False)
model.roberta.encoder.layer[-1].requires_grad_(True)
model.classifier.requires_grad_(True)
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=2e-5)
print('Training sentences:', len(train_data))
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))
before_weights = model.classifier.out_proj.weight.detach().clone()
training_losses = []
for epoch in range(3):
    model.train()
    model.roberta.eval()  # Frozen feature extraction has no dropout.
    model.roberta.encoder.layer[-1].train()
    total_loss = 0.0
    for batch in loader:
        optimizer.zero_grad()
        loss = model(**batch).loss  # Cross-entropy classification loss.
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch['labels'].size(0)
    average_loss = total_loss / len(train_data)
    training_losses.append(average_loss)
    print(f'Epoch {epoch + 1}/3 | training loss: {average_loss:.4f}')

weight_change = (model.classifier.out_proj.weight.detach() - before_weights).abs().max().item()
assert weight_change > 0, 'Weights did not change; check the training loop.'
print(f'Maximum classifier weight change: {weight_change:.8f}')
model.eval()
# All poster generation below now uses the adapted model.
emotion_model = pipeline('text-classification', model=model, tokenizer=tokenizer, device=-1)
model.save_pretrained('/content/MoodCanvas_Finetuned')
tokenizer.save_pretrained('/content/MoodCanvas_Finetuned')
print('Fine-tuning complete. Updated model saved to /content/MoodCanvas_Finetuned')


## 3. Match each emotion with a message and scene
The messages below are fixed sentences written for this project. The AI generates the background image, not the message.


In [ ]:
designs = {
    'sadness': ('Small steps still move you forward.', 'a peaceful lake at sunrise, soft blue and gold'),
    'joy': ('Keep growing. Keep going.', 'a bright sunny meadow with colorful flowers'),
    'anger': ('Pause. Breathe. Choose your next step.', 'calm ocean waves under a peaceful evening sky'),
    'fear': ('Courage begins with one small step.', 'a mountain path leading into warm morning light'),
    'surprise': ('Welcome a beautiful new possibility.', 'a glowing sky above a wide open landscape'),
    'disgust': ('Choose what brings you peace.', 'a clean green forest with gentle sunlight'),
    'neutral': ('A new day is a new beginning.', 'a quiet garden in soft morning light')
}


## 4. Create your poster
Change `mood_text` and run. We remove extra spaces, predict emotion, generate scenery, and add the message using Pillow. A fixed seed makes comparisons easier. Predictions can be wrong, especially for sarcasm or mixed emotions.


In [ ]:
mood_text = 'I am very happy today.'  # Change this sentence.
assert mood_text.strip(), 'Please enter a sentence.'
start = time.perf_counter()
clean_text = ' '.join(mood_text.split())
prediction = emotion_model(clean_text, truncation=True, max_length=128)[0]
emotion = prediction['label']
quote, scene = designs[emotion]
prompt = f'Watercolor landscape, {scene}, beautiful scenery, no text, no letters'
result = image_model(prompt, num_inference_steps=4, guidance_scale=0.0,
    height=512, width=512, generator=torch.Generator('cuda').manual_seed(42))
flags = getattr(result, 'nsfw_content_detected', None)
if flags and any(flags):
    raise RuntimeError('The image safety check flagged this output. Try a different seed.')
poster = Image.new('RGB', (768, 900), '#142332')
poster.paste(ImageOps.fit(result.images[0], (768, 600)), (0, 0))
draw = ImageDraw.Draw(poster)

# Find an available Colab font instead of assuming one fixed file path.
from pathlib import Path
font_paths = list(Path('/usr/share/fonts').rglob('*.ttf'))
def poster_font(size, bold=False):
    preferred = [p for p in font_paths if ('sans' in p.name.lower()) and
                 (('bold' in p.name.lower()) == bold)]
    for path in preferred + font_paths:
        try:
            return ImageFont.truetype(str(path), size)
        except OSError:
            continue
    return ImageFont.load_default(size=size)

font = poster_font(36, bold=True)
lines, line = [], ''
for word in quote.split():
    candidate = (line + ' ' + word).strip()
    if draw.textlength(candidate, font=font) > 660 and line:
        lines.append(line)
        line = word
    else:
        line = candidate
lines.append(line)
for i, line in enumerate(lines):
    draw.text((54, 650 + i * 52), line, fill='white', font=font)
small_font = poster_font(18)
draw.text((54, 850), 'MOODCANVAS AI', fill='#A8CADA', font=small_font)
poster.save('/content/MoodCanvas_Poster.png')
generation_seconds = time.perf_counter() - start
print(f'Predicted emotion: {emotion}. Model score: {prediction["score"]:.1%}')
print('This score is not a guarantee that the prediction is correct.')
print(f'Poster creation time: {generation_seconds:.1f} seconds')
display(poster)
files.download('/content/MoodCanvas_Poster.png')

## 5. Small demonstration test
We use **14 original example sentences**, two for each emotion. They are manually labeled for this classroom demo and are not a standard benchmark or proof of general performance. We do not fit or tune models on these 14 evaluation sentences. The 28 training sentences are separate. Compare the original and adapted models on the same evaluation set below.

**Accuracy:** fraction of predictions matching our labels. **Macro-F1:** average F1 across all seven emotions. Image generation is checked by showing the actual poster and reporting its generation time. These metrics do not measure whether the poster improves someone's mood.


In [ ]:
from sklearn.metrics import accuracy_score, f1_score
examples = [
    ('I am very sad and lonely today.', 'sadness'),
    ('I feel heartbroken after losing my friend.', 'sadness'),
    ('I am so happy about my wonderful results!', 'joy'),
    ('I feel delighted and cheerful today.', 'joy'),
    ('I am furious that you lied to me.', 'anger'),
    ('I am angry because someone broke my phone.', 'anger'),
    ('I am terrified of the dark.', 'fear'),
    ('I am scared about what might happen tomorrow.', 'fear'),
    ('Wow, I never expected this amazing gift!', 'surprise'),
    ('I am astonished by this unexpected news.', 'surprise'),
    ('That rotten food smells disgusting.', 'disgust'),
    ('I feel sick looking at that filthy rubbish.', 'disgust'),
    ('The meeting starts at three oclock.', 'neutral'),
    ('The book is on the table.', 'neutral')
]
# Verify separation and compare against the original pretrained checkpoint.
assert not set(train_texts) & {text for text, _ in examples}
baseline_answers = baseline_emotion_model([text for text, _ in examples], truncation=True, max_length=128)
baseline_predictions = [answer['label'] for answer in baseline_answers]
truth = [label for _, label in examples]
baseline_accuracy = accuracy_score(truth, baseline_predictions)
baseline_f1 = f1_score(truth, baseline_predictions, labels=list(designs), average='macro', zero_division=0)
print(f'Before fine-tuning: accuracy {baseline_accuracy:.1%}; macro-F1 {baseline_f1:.3f}')
answers = emotion_model([text for text, label in examples], truncation=True, max_length=128)
actual = [label for text, label in examples]
predicted = [answer['label'] for answer in answers]
accuracy = accuracy_score(actual, predicted)
macro_f1 = f1_score(actual, predicted, labels=list(designs), average='macro', zero_division=0)
for (text, label), guess in zip(examples, predicted):
    print(f'{label:9} | {guess:9} | {text}')
print(f'After fine-tuning demo accuracy: {accuracy:.1%}; macro-F1: {macro_f1:.3f}')
summary = {'training_size': 28, 'epochs': 3, 'batch_size': 4, 'learning_rate': 2e-5,
           'training_losses': training_losses, 'max_classifier_weight_change': weight_change,
           'baseline_accuracy': baseline_accuracy, 'baseline_macro_f1': baseline_f1, 'test_size': 14, 'accuracy': accuracy, 'macro_f1': macro_f1,
           'poster_emotion': globals().get('emotion'),
           'generation_seconds': globals().get('generation_seconds'),
           'note': 'Small manually labeled demonstration, not a benchmark; fine-tuning may not improve scores.'}
with open('/content/MoodCanvas_Results.json', 'w') as f:
    json.dump(summary, f, indent=2)
files.download('/content/MoodCanvas_Results.json')